# 10. Reflexión crítica

En este capítulo respondemos las preguntas de la guía con la evidencia cuantitativa de los capítulos anteriores. Las cifras se recalculan a partir de los resultados guardados; cada respuesta indica de qué sección proviene su evidencia.

In [1]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from src import config as C
from src import plots as P
from src import report as R
from src.persist import load_json

P.setup()
C.ensure_dirs()
pd.set_option("display.precision", 4)
pd.set_option("display.max_colwidth", 90)
print("Resultados:", C.RESULTS.relative_to(C.ROOT), "| prueba de humo:", C.SMOKE)

from src import metrics as M, spark_models as SP
enc = R.runs("scikit-learn", "encoding")
best_enc = enc.groupby("encoding")["val.roc_auc"].mean().idxmax()
cruce = pd.read_csv(C.SUMMARY / "punto_de_cruce.csv").set_index("modelo") if (C.SUMMARY / "punto_de_cruce.csv").exists() else pd.DataFrame()
p3 = pd.read_csv(C.SUMMARY / "p3_pruebas.csv").set_index("comparación") if (C.SUMMARY / "p3_pruebas.csv").exists() else pd.DataFrame()
t61 = pd.read_csv(C.SUMMARY / "tabla_6_1.csv")
sk_sc, sp_sc = R.runs("scikit-learn", "scale"), R.runs("PySpark", "scale")
grupos = C.column_groups()

Resultados: results | prueba de humo: False


## 10.1 ¿Qué entorno fue más rápido, en qué rango de volumen y por qué?

In [2]:
filas = []
for kind in ("lr_lbfgs", "mlp_lbfgs"):
    a = sk_sc[sk_sc["model"] == kind].groupby("n_train")["bench.fit_s"].mean()
    b = sp_sc[sp_sc["model"] == kind].groupby("n_train")["bench.fit_s"].mean()
    for n in sorted(set(a.index) & set(b.index)):
        filas.append({"modelo": R.label({"model": kind, "encoding": C.P1_ENCODING}), "filas": int(n), "scikit-learn (s)": a[n],
                      "PySpark (s)": b[n], "Spark / scikit-learn": b[n] / a[n]})
vel = pd.DataFrame(filas)
R.tabla(vel, ["modelo", "filas"], 3)
if len(cruce):
    display(cruce[["n*", "tipo"]])

*Sin datos: este experimento no se ejecutó dentro del límite de tiempo del cómputo (ver la sección de limitaciones y el anexo de reproducibilidad).*

In [3]:
txt = []
for m, g in vel.groupby("modelo"):
    rmin, rmax = g["Spark / scikit-learn"].min(), g["Spark / scikit-learn"].max()
    quien = "scikit-learn fue más rápido en todos los tamaños medidos" if rmin > 1 else (
            "Spark fue más rápido en todos los tamaños medidos" if rmax < 1 else "el entorno más rápido cambia con el tamaño")
    txt.append(f"**{m}:** {quien}; Spark tardó entre {rmin:.2f} y {rmax:.2f} veces lo que tardó scikit-learn "
               f"(de {R.num(g['filas'].min())} a {R.num(g['filas'].max())} filas).")
display(Markdown("\n\n".join(txt)))

*Sin datos: este experimento no se ejecutó dentro del límite de tiempo del cómputo (ver la sección de limitaciones y el anexo de reproducibilidad).*

**Explicación propuesta (hipótesis; no la medimos por separado en este equipo).** Tres mecanismos podrían explicar las diferencias de tiempo entre entornos en una sola máquina:

- **Sobrecarga fija de Spark.** Cada iteración de L-BFGS es un trabajo distribuido: el *driver* envía los pesos, programa tareas, recibe los gradientes parciales con `treeAggregate` y actualiza. Con pocas filas, esa coordinación domina el tiempo; con muchas, se amortiza entre más cómputo.
- **Representación de la entrada.** scikit-learn multiplica una matriz dispersa por los pesos y solo toca las 24 columnas activas de cada fila. El MLP de Spark apila la entrada en bloques densos de $d$ columnas, por lo que su costo por fila es unas $d/24 \approx 170$ veces mayor con $d = 2^{12}$ (sección 5.5). La regresión logística de Spark sí trabaja con vectores dispersos.
- **Paralelismo disponible.** scikit-learn usa un núcleo para la mayor parte del ajuste de la regresión logística y del MLP con L-BFGS, salvo las operaciones de álgebra lineal que NumPy delega en bibliotecas multihilo. Spark reparte las particiones entre todos los núcleos, cuya eficiencia debía medir la sección 7.4 (no se ejecutó por el límite de tiempo).

Si esta explicación es correcta, un punto de cruce, si existe, debería aparecer antes en la regresión logística (entrada dispersa en ambos entornos) que en el MLP (entrada densa solo en Spark).

## 10.2 ¿Cuál fue más preciso y calibrado, y es significativa la diferencia?

In [4]:
best = t61.sort_values("test.roc_auc|media", ascending=False).groupby("grupo").head(1)[["grupo", "modelo", "test.roc_auc|media", "test.log_loss|media", "test.ece|media"]]
display(best.rename(columns={"test.roc_auc|media": "ROC AUC", "test.log_loss|media": "log loss", "test.ece|media": "ECE"}).set_index("grupo").round(4))
y = R.y_split("test")
rows = []
for kind in ("lr_lbfgs", "mlp_lbfgs"):
    a = sk_sc[(sk_sc["model"] == kind) & (sk_sc["size"] == C.SPARK_MLP_SIZE)].set_index("seed")
    b = sp_sc[(sp_sc["model"] == kind) & (sp_sc["size"] == C.SPARK_MLP_SIZE)].set_index("seed")
    for s in sorted(set(a.index) & set(b.index)):
        pa, pb = R.preds(a.loc[s]), R.preds(b.loc[s])
        if pa is not None and pb is not None:
            d = M.delong_test(y, pa, pb)
            rows.append({"modelo": R.label({"model": kind, "encoding": C.P1_ENCODING}), "semilla": s, "Δ AUC": d["diff"], "p": d["p"],
                         "ECE scikit-learn": a.loc[s, "test.ece"], "ECE PySpark": b.loc[s, "test.ece"]})
eq = pd.DataFrame(rows)
res_eq = eq.groupby("modelo").agg(**{"Δ AUC medio (sklearn − Spark)": ("Δ AUC", "mean"), "semillas con p < 0.05": ("p", lambda s: int((s < C.ALPHA).sum())),
                                    "ECE scikit-learn": ("ECE scikit-learn", "mean"), "ECE PySpark": ("ECE PySpark", "mean")}) if len(eq) else pd.DataFrame()
display(res_eq.round(4))

,modelo,ROC AUC,log loss,ECE
grupo,,,,
scikit-learn · 1M,MLP · Hashing 2^14,0.7315,0.4055,0.0117
PySpark · 100k,Regresión logística · Hashing 2^12,0.7140,0.4175,0.0277


""


In [5]:
if len(res_eq):
    txt = " ".join(f"Con el mismo modelo ({m}), la diferencia media de AUC entre entornos es {r['Δ AUC medio (sklearn − Spark)']:+.4f}; "
                   f"DeLong la declara significativa en {int(r['semillas con p < 0.05'])} de {int((eq['modelo'] == m).sum())} semillas."
                   for m, r in res_eq.iterrows())
    txt += (" Ninguna diferencia supera δ, así que la precisión no decide la elección de entorno: los dos producen modelos equivalentes y la diferencia está en el costo."
            if (res_eq["Δ AUC medio (sklearn − Spark)"].abs() < C.DELTA_AUC).all() else
            " Al menos una diferencia supera δ; proviene de la implementación del optimizador y no de la distribución del cómputo.")
else:
    txt = "No hay corridas equivalentes completas para comparar la precisión entre entornos."
extra = (f" Entre los modelos de cada entorno, la mejor configuración de scikit-learn usa {R.num(C.SIZES[C.MAIN_SIZE])} filas, un tamaño que el MLP de Spark "
         f"no puede procesar con la memoria del driver de este equipo (sección 5.5); esa es la ventaja práctica de scikit-learn en precisión."
         if not SP.mlp_fits_memory(C.SIZES[C.MAIN_SIZE]) else "")
display(Markdown(txt + extra))

No hay corridas equivalentes completas para comparar la precisión entre entornos. Entre los modelos de cada entorno, la mejor configuración de scikit-learn usa 1 000 000 filas, un tamaño que el MLP de Spark no puede procesar con la memoria del driver de este equipo (sección 5.5); esa es la ventaja práctica de scikit-learn en precisión.

## 10.3 ¿Cuándo se justifica PySpark y cuándo es un sobrecosto?

**Se justifica cuando:**

- **Los datos no caben en la memoria de una máquina.** Criteo 1TB, con 24 días de registros, supera la RAM de cualquier portátil; Spark procesa por particiones y reparte el trabajo entre nodos de un clúster.
- **Hace falta tolerancia a fallos.** Si un nodo falla, Spark recalcula solo las particiones perdidas a partir del linaje del RDD; un proceso de scikit-learn que falla a las tres horas empieza de cero.
- **El dato ya vive en el ecosistema de Spark.** Si las impresiones llegan a un *data lake* en Parquet o Delta y el preprocesamiento ya es un trabajo de Spark, entrenar ahí evita mover los datos.
- **El preprocesamiento es el cuello de botella.** Las uniones, los conteos y las agregaciones sobre cientos de millones de filas se paralelizan bien en Spark.

**Es un sobrecosto cuando:**

- **Los datos caben en memoria.** Con Avazu completo (unos 40 millones de filas), Polars y DuckDB leen y agregan el Parquet en una sola máquina, y scikit-learn entrena con las muestras que necesitamos.
- **El modelo es un MLP con entrada de alta dimensión.** El MLP de Spark densifica la entrada, no admite *embeddings* ni *dropout* y no tiene parada temprana.
- **La tarea es iterativa y pequeña.** Cada iteración de un optimizador en Spark paga la coordinación del *driver*; con pocas filas, esa sobrecarga domina.

Las dos listas anteriores resumen criterios generales de la literatura y de la documentación de Spark. La evidencia de esta ejecución es la siguiente:


In [6]:
lec = load_json(C.COMMON / "lectura.json") if (C.COMMON / "lectura.json").exists() else {}
sp_main = R.runs("PySpark", "main")
sk_best = enc[enc["encoding"] == best_enc]
txt = []
pq = sorted(((k.split(" | ")[1], v["lectura_s"]) for k, v in lec.items() if k.startswith("Parquet")), key=lambda kv: kv[1])
if pq:
    sp_pq = dict(pq).get("PySpark")
    txt.append("**Lectura.** Leer el Parquet de entrenamiento (días 21 a 28) tomó " + ", ".join(f"{v:.1f} s con {k}" for k, v in pq) + "."
               + (f" PySpark tardó {sp_pq / pq[0][1]:.0f} veces lo que tardó el motor más rápido ({pq[0][0]})." if sp_pq and pq[0][0] != "PySpark" else ""))
for _, r in sp_main.iterrows():
    nombre = R.label({"model": r["model"], "encoding": r["encoding"]})
    txt.append(f"**{nombre} en Spark** ({R.num(r['n_train'])} filas, semilla {int(r['seed'])}): ajuste de {r['bench.fit_s']:.0f} s, "
               f"predicción del día 30 de {r['bench.predict_s']:.0f} s" + (f" ({r['bench.predict_s'] / 3600:.1f} h)" if r['bench.predict_s'] >= 360 else "") + f" y ROC AUC de prueba de {r['test.roc_auc']:.4f}.")
if len(sk_best):
    txt.append(f"**MLP de scikit-learn** ({C.ENCODING_LABELS[best_enc]}, {R.num(sk_best['n_train'].iat[0])} filas, {len(sk_best)} semillas): ajuste medio de "
               f"{sk_best['bench.fit_s'].mean() / 60:.0f} min, predicción media del día 30 de {sk_best['bench.predict_s'].mean():.0f} s y ROC AUC de prueba medio de {sk_best['test.roc_auc'].mean():.4f}.")
txt.append("Estas corridas no usan el mismo modelo ni el mismo tamaño en los dos entornos, así que describen el costo observado en este equipo y no "
           "responden por sí solas cuándo conviene Spark: esa respuesta exigía las curvas de escalado y el punto de cruce, que no alcanzaron a ejecutarse (sección 10.9).")
display(Markdown("\n\n".join(txt)))

**Lectura.** Leer el Parquet de entrenamiento (días 21 a 28) tomó 0.5 s con DuckDB, 1.6 s con Polars (streaming), 11.1 s con PySpark, 29.2 s con pandas (lotes de pyarrow). PySpark tardó 21 veces lo que tardó el motor más rápido (DuckDB).

**Regresión logística · Hashing 2^12 en Spark** (100 000 filas, semilla 0): ajuste de 11 s, predicción del día 30 de 17 s y ROC AUC de prueba de 0.7140.

**MLP · Hashing 2^12 en Spark** (100 000 filas, semilla 0): ajuste de 374 s, predicción del día 30 de 10805 s (3.0 h) y ROC AUC de prueba de 0.7109.

**MLP de scikit-learn** (Hashing 2^14, 1 000 000 filas, 5 semillas): ajuste medio de 39 min, predicción media del día 30 de 11 s y ROC AUC de prueba medio de 0.7315.

Estas corridas no usan el mismo modelo ni el mismo tamaño en los dos entornos, así que describen el costo observado en este equipo y no responden por sí solas cuándo conviene Spark: esa respuesta exigía las curvas de escalado y el punto de cruce, que no alcanzaron a ejecutarse (sección 10.9).

## 10.4 ¿Qué limitaciones impone el MLP nativo de Spark frente a una red con *embeddings*?


In [7]:
card = pd.Series(grupos["cardinalidad_tras_raras"])
k = 8
h = 100
emb_params = int(card.sum() * k)
d = C.SPARK_HASH_DIM
comp = pd.DataFrame({
    "parámetros de la primera capa": [d * h, int(card.sum()) * h, emb_params + len(card) * k * h],
    "entrada por fila": [f"vector denso de {d}", f"one-hot de {R.num(card.sum())} columnas", f"{len(card)} índices enteros"],
}, index=[f"MLP de Spark (hashing 2^12, {h} neuronas)", f"MLP con one-hot completo ({h} neuronas)", f"Red con embeddings (k = {k}, {h} neuronas)"])
display(comp)

,parámetros de la primera capa,entrada por fila
"MLP de Spark (hashing 2^12, 100 neuronas)",409600,vector denso de 4096
MLP con one-hot completo (100 neuronas),51173200,one-hot de 511 732 columnas
"Red con embeddings (k = 8, 100 neuronas)",4113056,24 índices enteros


Una red con *embeddings* asigna a cada categoría $v$ de la variable $j$ un vector aprendido $e_{j,v} \in \mathbb{R}^{k}$ y concatena los 24 vectores antes de las capas densas. Frente al MLP de Spark:

- **Sin colisiones.** Cada categoría frecuente tiene su propio vector; con hashing $2^{12}$, categorías distintas comparten columna.
- **Cómputo proporcional a las variables, no a la dimensión.** La entrada es una búsqueda en una tabla (24 índices por fila) seguida de una capa densa de $24k$ entradas; el MLP de Spark multiplica $d = 4096$ columnas por fila.
- **Regularización y optimización modernas.** *Dropout*, ReLU, Adam con minilotes y parada temprana, ausentes en `MultilayerPerceptronClassifier`.
- **Representaciones reutilizables.** Los *embeddings* agrupan categorías con comportamiento similar (por ejemplo, aplicaciones del mismo tipo) y sirven a otros modelos.

El costo es que una red con *embeddings* necesita un marco de aprendizaje profundo (PyTorch, TensorFlow) y, para entrenarse sobre un clúster de Spark, herramientas como `TorchDistributor`.

## 10.5 ¿Qué aporta LIME, qué no puede garantizar y cómo se compara con SHAP?

In [8]:
ex = load_json(C.SUMMARY / "explain.json")
lime = pd.read_json(C.SUMMARY / "lime_runs.json")
from src import explain as X
NS = max(C.LIME_NUM_SAMPLES)
d_ = lime[(lime["kernel"] == 0.75) & (lime["num_samples"] == NS)]
top5 = lambda dd: set(pd.Series(dd).abs().sort_values(ascending=False).index[:C.LIME_TOP_FEATURES])
jac = d_.assign(t=d_["pesos"].map(top5)).groupby("caso")["t"].apply(lambda s: X.mean_pairwise_jaccard(list(s)))
r2 = d_["r2"].median()
display(Markdown(
    f"Con la configuración por defecto, el $R^2$ mediano del sustituto local es {r2:.3f} y el Jaccard medio del top 5 entre semillas es {jac.mean():.3f} "
    f"(mínimo {jac.min():.2f} entre los {len(jac)} casos). Según los criterios de la sección 1.3, las explicaciones "
    + ("son estables y fieles." if (jac.mean() >= 0.8 and r2 >= 0.5) else
       ("son estables pero no fieles." if jac.mean() >= 0.8 else ("son fieles pero no estables." if r2 >= 0.5 else "no son estables ni fieles.")))))

Con la configuración por defecto, el $R^2$ mediano del sustituto local es 0.172 y el Jaccard medio del top 5 entre semillas es 0.737 (mínimo 0.55 entre los 20 casos). Según los criterios de la sección 1.3, las explicaciones no son estables ni fieles.

**Qué aporta LIME.** Traduce una predicción de un modelo opaco en una lista corta de valores concretos ("`banner_pos=1` sube la probabilidad"), sin acceder a los pesos del modelo. Sirve para revisar errores confiados, detectar variables con efectos inesperados y comunicar el comportamiento del modelo a quien no lo construyó.

**Qué no puede garantizar.**

- **Fidelidad global.** El sustituto lineal es válido en el vecindario elegido; un $R^2$ bajo indica que el modelo no es lineal ni siquiera localmente.
- **Estabilidad.** Las perturbaciones son aleatorias; dos ejecuciones pueden dar listas distintas (sección 8.5).
- **Independencia de la configuración.** El ancho del núcleo define qué es "local" y cambia las explicaciones sin un criterio objetivo para elegirlo.
- **Causalidad.** Un peso alto indica que el modelo usa la variable, no que la variable cause el clic.
- **Distribución realista.** LIME muestrea cada variable por separado y evalúa al modelo en combinaciones que no existen en los datos (por ejemplo, un `site_id` con un `site_domain` que no le corresponde).

**Frente a SHAP.** Los valores de Shapley tienen propiedades formales (eficiencia, simetría, aditividad) que LIME no tiene, y con suficientes evaluaciones convergen a un valor único para cada caso. Comparten con LIME la evaluación en combinaciones poco realistas cuando el fondo es independiente, y son más costosos. En nuestra comparación (sección 8.6), el acuerdo entre ambos métodos sobre las variables principales es la evidencia más útil: cuando coinciden, la explicación es más confiable que cuando solo la da uno.

## 10.6 Amenazas a la validez

In [9]:
psi = pd.read_csv(C.EDA_OUT / "psi.csv")
deriva = psi[(psi["psi"] >= 0.1) & (psi["variable"] != "dia_semana")]["variable"].tolist()
amenazas = [
    ("Deriva temporal", f"El CTR cambia entre días (sección 2.2) y {len(deriva)} variables tienen PSI ≥ 0.1 entre entrenamiento y prueba (sección 2.8). "
     "Los resultados describen el desempeño en un día de prueba posterior a la ventana de entrenamiento, no en un período cualquiera."),
    ("Fuga de información", "La partición temporal, el ajuste de todas las transformaciones con entrenamiento, el target encoding fuera de pliegue y la huella del conjunto de prueba (sección 3.7) cierran las vías de fuga que identificamos. "
     "La parada temprana interna de `MLPClassifier` usa una partición aleatoria de entrenamiento, lo que no filtra información de prueba pero hace optimista su criterio."),
    ("Anonimización", "C1 y C14–C21 son códigos sin significado conocido y los identificadores están cifrados; las explicaciones del capítulo 8 dicen qué variables usa el modelo, pero no permiten interpretarlas en términos del negocio."),
    ("Sesgos de muestreo", "Las muestras se eligen con una clave uniforme y son representativas de la ventana de entrenamiento. Avazu, en cambio, contiene solo las impresiones que su sistema decidió mostrar en una plataforma y diez días; no representa otras plataformas ni otras épocas del año."),
    ("Dependencia entre impresiones", "Un mismo dispositivo o IP genera muchas impresiones; los intervalos bootstrap por fila y las pruebas de DeLong y McNemar suponen independencia y son optimistas. La prueba $t$ entre semillas no depende de ese supuesto."),
    ("Validez de la comparación de entornos", "Los tiempos dependen del equipo, de la versión de las bibliotecas y de la carga del sistema durante la medición. Spark en modo local no reproduce la red ni la tolerancia a fallos de un clúster."),
    ("Presupuesto de cómputo", "Los ajustes que superaron el presupuesto de tiempo o de memoria se omitieron y se reportan con su estimación; las curvas de escalado de esos tamaños son extrapolaciones."),
]
display(Markdown("| Amenaza | Evaluación |\n|---|---|\n" + "\n".join(f"| {a} | {b} |" for a, b in amenazas)))

| Amenaza | Evaluación |
|---|---|
| Deriva temporal | El CTR cambia entre días (sección 2.2) y 10 variables tienen PSI ≥ 0.1 entre entrenamiento y prueba (sección 2.8). Los resultados describen el desempeño en un día de prueba posterior a la ventana de entrenamiento, no en un período cualquiera. |
| Fuga de información | La partición temporal, el ajuste de todas las transformaciones con entrenamiento, el target encoding fuera de pliegue y la huella del conjunto de prueba (sección 3.7) cierran las vías de fuga que identificamos. La parada temprana interna de `MLPClassifier` usa una partición aleatoria de entrenamiento, lo que no filtra información de prueba pero hace optimista su criterio. |
| Anonimización | C1 y C14–C21 son códigos sin significado conocido y los identificadores están cifrados; las explicaciones del capítulo 8 dicen qué variables usa el modelo, pero no permiten interpretarlas en términos del negocio. |
| Sesgos de muestreo | Las muestras se eligen con una clave uniforme y son representativas de la ventana de entrenamiento. Avazu, en cambio, contiene solo las impresiones que su sistema decidió mostrar en una plataforma y diez días; no representa otras plataformas ni otras épocas del año. |
| Dependencia entre impresiones | Un mismo dispositivo o IP genera muchas impresiones; los intervalos bootstrap por fila y las pruebas de DeLong y McNemar suponen independencia y son optimistas. La prueba $t$ entre semillas no depende de ese supuesto. |
| Validez de la comparación de entornos | Los tiempos dependen del equipo, de la versión de las bibliotecas y de la carga del sistema durante la medición. Spark en modo local no reproduce la red ni la tolerancia a fallos de un clúster. |
| Presupuesto de cómputo | Los ajustes que superaron el presupuesto de tiempo o de memoria se omitieron y se reportan con su estimación; las curvas de escalado de esos tamaños son extrapolaciones. |

## 10.7 Respuesta a las preguntas de investigación

In [10]:
filas = []
# P1
if len(cruce):
    txt = []
    for m, r in cruce.iterrows():
        if pd.notna(r["n*"]) and str(r["tipo"]).startswith("medido"):
            txt.append(f"{m}: cruce medido en ≈ {R.num(r['n*'])} filas")
        elif pd.notna(r["n*"]):
            txt.append(f"{m}: sin cruce en el rango medido; extrapolado a ≈ {R.num(r['n*'])} filas")
        else:
            txt.append(f"{m}: sin cruce")
    decision = ("H1 se sostiene para " + ", ".join(m for m, r in cruce.iterrows() if pd.notna(r["n*"]) and str(r["tipo"]).startswith("medido"))
                if any(pd.notna(r["n*"]) and str(r["tipo"]).startswith("medido") for _, r in cruce.iterrows()) else
                "No observamos un punto de cruce dentro de los tamaños viables en este equipo; scikit-learn fue más eficiente en todo el rango")
    filas.append(("P1. Punto de cruce", "; ".join(txt), decision))
else:
    filas.append(("P1. Punto de cruce", "Sin corridas de escalado de PySpark (sección 10.9)", "No evaluada: faltan datos para decidir sobre H1"))
# P2
piv = enc.pivot(index="seed", columns="encoding", values="test.roc_auc")
if {"hash_14", "freq"} <= set(piv):
    t = M.paired_t(piv["hash_14"].values, piv["freq"].values)
    mem = enc.groupby("encoding")["bench.matrix_mb"].mean()
    eqv = t.get("lo", np.nan) > -C.DELTA_AUC and t.get("hi", np.nan) < C.DELTA_AUC
    filas.append(("P2. Codificación", f"Hashing 2^14 − frecuencia: ΔAUC = {t['diff']:+.4f} [{t.get('lo', np.nan):.4f}, {t.get('hi', np.nan):.4f}]; memoria relativa {mem['hash_14'] / mem['freq']:.2f}; mejor en validación: {C.ENCODING_LABELS[best_enc]}",
                  ("H2 se sostiene: equivalencia práctica" + (" con menos memoria" if mem["hash_14"] < mem["freq"] else ", pero sin ahorro de memoria")) if eqv else
                  "H2 no se sostiene: no hay equivalencia práctica dentro de ±δ"))
# P3
if len(p3):
    gana = p3[p3["significativa y ≥ δ"].astype(bool)]
    filas.append(("P3. Aporte del MLP", "; ".join(f"{i.split(' − ')[1]}: Δ = {r['diff']:+.4f}, p Holm = {r['p Holm']:.3g}" for i, r in p3.iterrows()),
                  (f"Se rechaza H0 frente a {len(gana)} de {len(p3)} líneas base" if len(gana) else "No se rechaza H0 con el criterio declarado")))
# P4
filas.append(("P4. Calidad de LIME", f"R² mediano = {r2:.3f}; Jaccard medio del top 5 = {jac.mean():.3f}",
              "Estables y fieles" if (jac.mean() >= 0.8 and r2 >= 0.5) else "No cumplen ambos criterios declarados"))
display(Markdown("| Pregunta | Evidencia | Decisión |\n|---|---|---|\n" + "\n".join(f"| {a} | {b} | {c} |" for a, b, c in filas)))

| Pregunta | Evidencia | Decisión |
|---|---|---|
| P1. Punto de cruce | Sin corridas de escalado de PySpark (sección 10.9) | No evaluada: faltan datos para decidir sobre H1 |
| P2. Codificación | Hashing 2^14 − frecuencia: ΔAUC = +0.0682 [0.0519, 0.0846]; memoria relativa 0.20; mejor en validación: Hashing 2^14 | H2 no se sostiene: no hay equivalencia práctica dentro de ±δ |
| P3. Aporte del MLP | Regresión logística · Hashing 2^14: Δ = +0.0042, p Holm = 0.0314; Regresión logística · Frecuencia: Δ = +0.0538, p Holm = 1.44e-05; Gradient boosting · Frecuencia: Δ = +0.0069, p Holm = 0.000588 | Se rechaza H0 frente a 2 de 3 líneas base |
| P4. Calidad de LIME | R² mediano = 0.172; Jaccard medio del top 5 = 0.737 | No cumplen ambos criterios declarados |

## 10.8 Conclusión

In [11]:
equiv = len(res_eq) > 0 and bool((res_eq["Δ AUC medio (sklearn − Spark)"].abs() < C.DELTA_AUC).all())
cruce_medido = len(cruce) > 0 and any(pd.notna(r["n*"]) and str(r["tipo"]).startswith("medido") for _, r in cruce.iterrows())
txt = ("En una sola máquina y con Avazu, la herramienta adecuada depende de la etapa. Para leer y agregar los datos, el Parquet con DuckDB o Polars "
       "resuelve el volumen completo sin un clúster. Para entrenar, scikit-learn trabaja con la entrada dispersa y admite muestras más grandes que el "
       "MLP nativo de Spark en la misma memoria")
if len(res_eq) == 0:
    txt += ". En esta ejecución no hay corridas pareadas del mismo modelo en ambos entornos, así que no podemos afirmar si su calidad es equivalente (sección 10.2). "
elif equiv:
    txt += ", y los dos entornos producen modelos de calidad equivalente cuando entrenan el mismo modelo. "
else:
    txt += "; cuando ambos entrenan el mismo modelo, sus AUC difieren en más de δ por diferencias de implementación. "
if len(cruce) == 0:
    txt += ("No medimos el punto de cruce: las corridas de escalado de PySpark no se ejecutaron dentro del límite de tiempo (sección 10.9), "
            "por lo que no podemos afirmar ni descartar que Spark supere a scikit-learn en tiempo a mayor volumen. ")
elif cruce_medido:
    txt += "Spark supera a scikit-learn en tiempo solo a partir del volumen de cruce medido en el capítulo 7. "
else:
    txt += "En el rango que este equipo permite medir, Spark no supera a scikit-learn en tiempo de entrenamiento. "
txt += ("Fuera de lo que medimos, la literatura indica que Spark se vuelve necesario cuando el volumen supera la memoria de una máquina, como en Criteo 1TB; en ese escenario, el MLP nativo "
        "debería reemplazarse por una red con embeddings entrenada de forma distribuida.")
display(Markdown(txt))

En una sola máquina y con Avazu, la herramienta adecuada depende de la etapa. Para leer y agregar los datos, el Parquet con DuckDB o Polars resuelve el volumen completo sin un clúster. Para entrenar, scikit-learn trabaja con la entrada dispersa y admite muestras más grandes que el MLP nativo de Spark en la misma memoria. En esta ejecución no hay corridas pareadas del mismo modelo en ambos entornos, así que no podemos afirmar si su calidad es equivalente (sección 10.2). No medimos el punto de cruce: las corridas de escalado de PySpark no se ejecutaron dentro del límite de tiempo (sección 10.9), por lo que no podemos afirmar ni descartar que Spark supere a scikit-learn en tiempo a mayor volumen. Fuera de lo que medimos, la literatura indica que Spark se vuelve necesario cuando el volumen supera la memoria de una máquina, como en Criteo 1TB; en ese escenario, el MLP nativo debería reemplazarse por una red con embeddings entrenada de forma distribuida.

## 10.9 Limitaciones computacionales de esta ejecución

Ejecutamos todo el experimento en un solo computador portátil, sin GPU utilizable por scikit-learn ni por Spark MLlib, y con una fecha de entrega fija. Para no perder lo ya calculado, organizamos el cómputo por prioridades: primero la búsqueda de hiperparámetros y la comparación de codificaciones en scikit-learn, después los modelos de PySpark, el *benchmark* de lectura y la interpretabilidad, y al final las ablaciones. Cada etapa recibió una hora límite; al alcanzarla, dejó de iniciar corridas nuevas y conservó todas las terminadas. La tabla siguiente compara lo planeado con lo que alcanzamos a ejecutar, y el texto que la acompaña resume dónde se concentró el tiempo. Donde falta un experimento, el informe lo indica con una nota en lugar de un resultado.

In [12]:
from pathlib import Path as _P
sk_all = R.runs("scikit-learn", None, include_skipped=True)
sp_all = R.runs("PySpark", None, include_skipped=True)
n_sz = sum(C.SEEDS_BY_SIZE.values()); ns = len(C.SEEDS); nc = len(C.SPARK_CORES_SCALING)
plan = {("scikit-learn", "encoding"): len(C.ENCODINGS) * ns, ("scikit-learn", "baselines"): 3 * ns,
        ("scikit-learn", "scale"): 2 * n_sz, ("scikit-learn", "ablation_weight"): ns,
        ("scikit-learn", "ablation_cross"): ns, ("scikit-learn", "ablation_depth"): ns * len(C.DEPTH_ABLATION),
        ("PySpark", "main"): 5 * ns, ("PySpark", "scale"): 2 * n_sz, ("PySpark", "strong"): 2 * nc, ("PySpark", "weak"): 2 * nc}
nombres = {"encoding": "Codificaciones (P2)", "baselines": "Líneas base (P3)", "scale": "Escalado por tamaño (P1)",
           "ablation_weight": "Ablación: pesos por clase", "ablation_cross": "Ablación: cruces",
           "ablation_depth": "Ablación: profundidad", "main": "Modelos principales", "strong": "Escalamiento fuerte",
           "weak": "Escalamiento débil"}
filas = []
for (env, exp), n_plan in plan.items():
    df = sk_all if env == "scikit-learn" else sp_all
    g = df[df["exp"] == exp]
    ok = g[~g["omitido"]]
    filas.append({"entorno": env, "experimento": nombres[exp], "planeadas": n_plan, "terminadas": len(ok),
                  "omitidas (memoria o presupuesto)": int(g["omitido"].sum()),
                  "sin ejecutar (límite de tiempo)": max(n_plan - len(g), 0),
                  "horas de entrenamiento": ok["bench.fit_s"].sum() / 3600})
lim = pd.DataFrame(filas)
R.tabla(lim, ["entorno", "experimento"], 2)

txt = []
opt = []
for f in sorted(C.COMMON.glob("optuna_*_trials.csv")):
    t = pd.read_csv(f)
    if "duration" in t:
        h = pd.to_timedelta(t["duration"]).dt.total_seconds().sum() / 3600
        opt.append((f.stem.replace("optuna_", "").replace("_trials", ""), len(t), h))
if opt:
    txt.append("La búsqueda bayesiana con Optuna consumió " + "; ".join(
        f"{h:.1f} h en {n} pruebas para {k.replace('_', ' ')}" for k, n, h in opt) + ".")
enc_all = sk_all[(sk_all["exp"] == "encoding") & ~sk_all["omitido"]]
if len(enc_all):
    m = enc_all.groupby("encoding")["bench.fit_s"].mean().sort_values()
    txt.append(f"En la comparación de codificaciones, el tiempo medio de entrenamiento del MLP con {R.num(C.SIZES[C.MAIN_SIZE])} filas fue de "
               + ", ".join(f"{v / 60:.0f} min con {C.ENCODING_LABELS.get(k, k)}" for k, v in m.items()) + ".")
    if {"hash_14", "hash_16"} <= set(m.index):
        auc = enc_all.groupby("encoding")["test.roc_auc"].mean()
        txt.append(f"Pasar de 2^14 a 2^16 columnas multiplicó el tiempo por {m['hash_16'] / m['hash_14']:.1f} y cambió el ROC AUC medio "
                   f"en {auc['hash_16'] - auc['hash_14']:+.4f}: el costo crece con la dimensión de la entrada sin una ganancia proporcional.")
reg = C.COMMON / "limite_tiempo.json"
if reg.exists():
    cortes = load_json(reg)
    txt.append("Las etapas " + ", ".join(f"**{k}** (límite {v['limite']})" for k, v in cortes.items())
               + " alcanzaron su hora límite; lo que aparece como *sin ejecutar* en la tabla quedó fuera por esa razón y no por un error.")
else:
    txt.append("Ninguna etapa alcanzó su hora límite: ejecutamos el plan completo.")
hwb = C.COMMON / "hardware_benchmark.json"
if hwb.exists():
    h1, h2 = load_json(C.COMMON / "hardware.json")["hardware"], load_json(hwb)["hardware"]
    txt.append(f"Para cumplir el plazo repartimos el cómputo entre dos equipos: scikit-learn (codificaciones, líneas base, "
               f"interpretabilidad y ablaciones) corrió en uno con {h1['nucleos_logicos']} núcleos lógicos y {h1['ram_gb']} GB de RAM, "
               f"y todo lo que depende del hardware (PySpark, escalado de scikit-learn, escalamiento y lectura) en otro con "
               f"{h2['nucleos_logicos']} núcleos lógicos y {h2['ram_gb']} GB. Los tiempos se comparan solo dentro de ese segundo "
               "equipo; las métricas de calidad no dependen del hardware.")
txt.append("Estas limitaciones no cambian la validez de lo que sí reportamos (cada resultado usa la misma partición temporal, "
           "las mismas semillas y la misma evaluación en el día 30), pero sí reducen el alcance: con menos semillas o sin algunos "
           "tamaños, los intervalos de confianza son más anchos y algunas comparaciones quedan sin prueba estadística.")
display(Markdown("\n\n".join(txt)))

planeadas  terminadas  \
entorno      experimento                                        
scikit-learn Codificaciones (P2)               25          25   
             Líneas base (P3)                  15          15   
             Escalado por tamaño (P1)          32          19   
             Ablación: pesos por clase          5           1   
             Ablación: cruces                   5           1   
             Ablación: profundidad             20           0   
PySpark      Modelos principales               25           2   
             Escalado por tamaño (P1)          32           0   
             Escalamiento fuerte                8           0   
             Escalamiento débil                 8           0   

                                        omitidas (memoria o presupuesto)  \
entorno      experimento                                                   
scikit-learn Codificaciones (P2)                                       0   
             Líneas base (P3)                                          0   
             Escalado por tamaño (P1)                                  0   
             Ablación: pesos por clase                                 0   
             Ablación: cruces                                          0   
             Ablación: profundidad                                     0   
PySpark      Modelos principales                                       0   
             Escalado por tamaño (P1)                                  0   
             Escalamiento fuerte                                       0   
             Escalamiento débil                                        0   

                                        sin ejecutar (límite de tiempo)  \
entorno      experimento                                                  
scikit-learn Codificaciones (P2)                                      0   
             Líneas base (P3)                                         0   
             Escalado por tamaño (P1)                                13   
             Ablación: pesos por clase                                4   
             Ablación: cruces                                         4   
             Ablación: profundidad                                   20   
PySpark      Modelos principales                                     23   
             Escalado por tamaño (P1)                                32   
             Escalamiento fuerte                                      8   
             Escalamiento débil                                       8   

                                        horas de entrenamiento  
entorno      experimento                                        
scikit-learn Codificaciones (P2)                         17.71  
             Líneas base (P3)                             0.20  
             Escalado por tamaño (P1)                     0.09  
             Ablación: pesos por clase                    0.62  
             Ablación: cruces                             0.60  
             Ablación: profundidad                        0.00  
PySpark      Modelos principales                          0.11  
             Escalado por tamaño (P1)                     0.00  
             Escalamiento fuerte                          0.00  
             Escalamiento débil                           0.00

La búsqueda bayesiana con Optuna consumió 0.2 h en 15 pruebas para hgb freq; 0.0 h en 15 pruebas para lr freq; 0.0 h en 15 pruebas para lr hash 14; 6.4 h en 30 pruebas para mlp hash 14.

En la comparación de codificaciones, el tiempo medio de entrenamiento del MLP con 1 000 000 filas fue de 5 min con Target encoding OOF, 6 min con Frecuencia, 12 min con Hashing 2^12, 39 min con Hashing 2^14, 151 min con Hashing 2^16.

Pasar de 2^14 a 2^16 columnas multiplicó el tiempo por 3.9 y cambió el ROC AUC medio en -0.0015: el costo crece con la dimensión de la entrada sin una ganancia proporcional.

Las etapas **spark** (límite 2026-11-11 06:00), **escala_sklearn** (límite 2026-11-11 07:00), **ablaciones** (límite 2026-11-11 08:15) alcanzaron su hora límite; lo que aparece como *sin ejecutar* en la tabla quedó fuera por esa razón y no por un error.

Estas limitaciones no cambian la validez de lo que sí reportamos (cada resultado usa la misma partición temporal, las mismas semillas y la misma evaluación en el día 30), pero sí reducen el alcance: con menos semillas o sin algunos tamaños, los intervalos de confianza son más anchos y algunas comparaciones quedan sin prueba estadística.

### Control de cambios de la ejecución

Esta lista registra, en orden, las decisiones que tomamos durante la ejecución en el equipo usado (anexo A) para ajustar el plan al tiempo y al hardware disponibles. Las horas son las del registro `results/logs/todo.log`.

**Recursos del equipo.** Un portátil con 31.3 GB de RAM. Tiene una GPU integrada AMD Radeon y una NVIDIA GeForce RTX, pero ninguna se usó: `MLPClassifier` de scikit-learn y Spark MLlib solo entrenan en CPU, y el Administrador de tareas mostró ambas GPU entre 0 % y 4 % de uso durante el entrenamiento.

| # | Fecha y hora | Situación observada | Decisión o cambio | Efecto en el informe |
|---|---|---|---|---|
| 1 | 9 nov, 22:57 | Plan inicial: todo el experimento con `python -m src.runner all` (Optuna con 30 intentos para el MLP y 15 por línea base sobre 500 mil filas; cinco codificaciones con cinco semillas sobre un millón de filas; líneas base; escalado de 100 mil a 10 millones; Spark completo; lectura; LIME y SHAP; ablaciones). | Se lanzó la ejecución completa sin límites de tiempo. | — |
| 2 | 10 nov | La búsqueda de Optuna del MLP tomó 6.4 h. El MLP con hashing $2^{16}$ tardó entre 2.1 y 3.1 h por semilla. Tras cerca de 24 h, la etapa de scikit-learn seguía en curso y Spark no había empezado. | Revisamos el registro de tiempos para estimar lo que faltaba y concluimos que el plan completo no cabía antes de la entrega. | Sección 10.9: la comparación de codificaciones sumó 17.7 h de entrenamiento. |
| 3 | 10 nov | Un corte de la ejecución habría obligado a reentrenar los modelos para la interpretabilidad. | Guardamos cada modelo y cada preprocesador entrenado en disco con escritura atómica, para reutilizarlos sin reentrenar. | La interpretabilidad (capítulo 8) usa el MLP ya entrenado de la semilla mediana. |
| 4 | 10 nov | Sin tiempo para el plan completo. | Asignamos una hora límite a cada etapa y un orden de prioridad: preparación → scikit-learn → interpretabilidad → Spark → escalado de scikit-learn → lectura → ablaciones. Al llegar a su límite, una etapa deja de iniciar corridas nuevas y conserva las terminadas. | Lo que no alcanzó a ejecutarse aparece como *sin ejecutar* en la tabla de la sección 10.9. |
| 5 | 10 nov | Algunos cuadernos fallaban si un experimento no existía. | Hicimos que cada análisis sin datos muestre una nota en lugar de un error, y que el texto de conclusiones distinga entre lo medido y lo no medido. | Notas *Sin datos* en los capítulos 7 y 10. |
| 6 | 10 nov, 22:54–22:59 | Estaban terminadas 19 de las 25 corridas de codificación (hashing $2^{12}$, $2^{14}$ y $2^{16}$ con cinco semillas, y frecuencia con cuatro). | Detuvimos la ejecución y la relanzamos con los límites: scikit-learn hasta las 01:30, Spark hasta las 06:00, escalado de scikit-learn hasta las 07:00 y ablaciones hasta las 08:15 del 11 de noviembre. Se conservaron las 19 corridas terminadas. | Ninguna corrida terminada se perdió ni se repitió. |
| 7 | 10 nov, 23:56 | La etapa de scikit-learn terminó en 56.7 min, antes de su límite. | — | Codificaciones (P2) y líneas base (P3) completas con cinco semillas. |
| 8 | 10 nov, 23:57–11 nov, 03:17 | Spark no pudo cargar la biblioteca nativa de álgebra lineal (`libopenblas.dll`) y usó la implementación en Java, más lenta. La memoria del *driver* se fijó en 16 GB de los 31.3 GB del equipo; durante el ajuste, el equipo llegó al 95 % de uso de memoria y la CPU estuvo cerca del 11 %. | Mantuvimos la configuración para no arriesgar fallos por falta de memoria. | La rejilla del MLP de Spark (32 combinaciones sobre 20 mil filas) tomó 3 h 20 min. |
| 9 | 11 nov, 03:18–06:52 | El MLP de Spark con 100 mil filas (semilla 0) se ajustó en 374 s, pero la predicción del día 30 (4.2 millones de filas, entrada densa de 4096 columnas) tomó 10 805 s. | La etapa llegó a su límite al terminar esa corrida y no inició las demás. | Spark quedó con 2 de 25 corridas principales (regresión logística y MLP, semilla 0) y sin escalado por tamaño, escalamiento fuerte ni débil. |
| 10 | 11 nov, 06:52–07:00 | Escalado de scikit-learn con su límite. | — | 19 de 32 corridas: regresión logística de 100 mil a 10 millones; MLP solo con 100 mil (tres semillas). |
| 11 | 11 nov, 07:00–07:04 | Benchmark de lectura. | — | Completo (sección 7.1). |
| 12 | 11 nov, 07:04–08:17 | Ablaciones con su límite. | — | Ponderación de clases y cruces con una semilla cada una; profundidad sin ejecutar. |
| 13 | 11 nov, 08:18–20:42 | La compilación automática del informe falló porque la terminal que la lanzó no encontraba Java. | Repetimos la compilación desde una terminal con Java disponible. | Cuadernos e informe generados en 12 min. |
| 14 | 11 nov, noche | Revisión del informe con los resultados reales. | Reemplazamos afirmaciones que el código escribía por defecto sin datos que las respaldaran (secciones 10.7 y 10.8), marcamos como hipótesis las explicaciones que no medimos y corregimos leyendas, etiquetas y tablas de las figuras. | Versión final del informe. |

**Alcance de los resultados.** Las preguntas P2 (codificación), P3 (aporte del MLP) y P4 (LIME) se responden con cinco semillas por configuración. Para P1, el informe presenta el método de medición (sección 7.3) y la evidencia de costo obtenida en este equipo (secciones 7.1, 7.2 y 10.3); la medición del punto de cruce y de las curvas de escalamiento queda como el paso siguiente del estudio. La comparación de calidad entre entornos se apoya en una semilla de Spark con 100 mil filas.
